In [ ]:
import hashlib

def hash_node(left, right):
    # Combine and hash two nodes
    combined = f"{left}{right}".encode('utf-8')
    return hashlib.sha256(combined).hexdigest()

class SimpleMerkleTree:
    def __init__(self, leaves):
        self.leaves = leaves
        self.tree = [leaves]
        self._build_tree()

    def _build_tree(self):
        # Build the tree level by level from bottom to top
        current_level = self.leaves
        while len(current_level) > 1:
            next_level = []
            # Process pairs of nodes
            for i in range(0, len(current_level), 2):
                left = current_level[i]
                # Handle odd number of nodes by duplicating the last one
                right = current_level[i+1] if i+1 < len(current_level) else left
                next_level.append(hash_node(left, right))
            self.tree.append(next_level)
            current_level = next_level

    def get_root(self):
        # The root is the only element in the highest level
        return self.tree[-1][0]

    def get_path(self, index):
        # Generate the Merkle path for a specific leaf index
        path = []
        for level in range(len(self.tree) - 1):
            is_right_node = index % 2 != 0
            sibling_index = index - 1 if is_right_node else index + 1
            
            # Handle edge case where sibling index is out of bounds
            if sibling_index < len(self.tree[level]):
                path.append({
                    "direction": "left" if is_right_node else "right",
                    "hash": self.tree[level][sibling_index]
                })
            index //= 2
        return path

if __name__ == "__main__":
    # 1. Initialize leaves (e.g., hashed student IDs)
    student_hashes = [
        hashlib.sha256(b"1120001").hexdigest(),
        hashlib.sha256(b"1120002").hexdigest(),
        hashlib.sha256(b"1120003").hexdigest(),
        hashlib.sha256(b"1120004").hexdigest()
    ]

    # 2. Build the tree
    merkle_tree = SimpleMerkleTree(student_hashes)
    
    # 3. Get the Global Root (This goes to the Laravel backend)
    global_root = merkle_tree.get_root()
    print(f"Global Root: {global_root}\n")

    # 4. Get the Path for Student 1120003 (Index 2)
    target_index = 2
    path = merkle_tree.get_path(target_index)
    
    # This path is kept local and fed into the ZKP circuit
    print(f"Student {target_index} (1120003) Merkle Path:")
    for step in path:
        print(f"Sibling ({step['direction']}): {step['hash']}")